# Cas pratique n°2 — Titanic : modèle enrichi, recherche d'hyperparamètres, soumission

**Jour 2 — chapitre 03 : Feature engineering (suite) / cas pratique n°2**

## Mise en situation

En reprenant les features nettoyées (notebook 03) et enrichies (notebook 04), on entraîne
ici un nouveau modèle et on le compare au premier modèle du notebook 01. On cherche
ensuite ses meilleurs hyperparamètres, puis on génère un fichier de soumission au format
Kaggle.


In [1]:
import pandas as pd
import seaborn as sns

titanic_raw = sns.load_dataset("titanic")
titanic = titanic_raw.rename(columns={
    "survived": "Survived",
    "pclass": "Pclass",
    "sex": "Sex",
    "age": "Age",
    "sibsp": "SibSp",
    "parch": "Parch",
    "fare": "Fare",
    "embarked": "Embarked",
})[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

# Nettoyage (TP "valeurs manquantes", notebook 03)
age_median_par_groupe = titanic.groupby(["Sex", "Pclass"])["Age"].transform("median")
titanic["Age"] = titanic["Age"].fillna(age_median_par_groupe).fillna(titanic["Age"].median())
titanic["Embarked"] = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])

# Feature engineering (TP "feature engineering", notebook 04)
titanic = pd.get_dummies(titanic, columns=["Sex", "Embarked"], drop_first=True)
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1
titanic["IsAlone"] = (titanic["FamilySize"] == 1).astype(int)

# PassengerId synthetique pour le format de soumission (le dataset seaborn n'en fournit pas)
titanic = titanic.reset_index(drop=True)
titanic.insert(0, "PassengerId", titanic.index + 1)

features = ["Pclass", "Age", "Fare", "FamilySize", "IsAlone", "Sex_male"]
features = [c for c in features if c in titanic.columns]

X = titanic[features]
y = titanic["Survived"]

titanic.head()


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,FamilySize,IsAlone
0,1,0,3,22.0,1,0,7.2500,True,False,True,2,0
1,2,1,1,38.0,1,0,71.2833,False,False,False,2,0
2,3,1,3,26.0,0,0,7.9250,False,False,True,1,1
3,4,1,1,35.0,1,0,53.1000,False,False,True,2,0
4,5,0,3,35.0,0,0,8.0500,True,False,True,1,1


## Étape 1 — Un modèle sur les features nettoyées et enrichies

**Question de réflexion :** le nettoyage (`Age` imputé) et les nouvelles variables
(`FamilySize`, `IsAlone`) améliorent-ils le premier modèle du notebook 01 ? Comment
comparer les deux modèles équitablement ?


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## Suivi des expériences avec MLflow

Plutôt que de garder une trace manuelle des scores dans un carnet ou un tableur (voir la
check-list du jour 3 : « a-t-on conservé une trace des expériences menées ? »), on utilise
ici **MLflow** pour enregistrer automatiquement les paramètres et les métriques de chaque
essai.

Ceci suppose qu'un serveur de tracking MLflow tourne en local, lancé au préalable avec :

```bash
mlflow server --host 127.0.0.1 --port 5001 \
    --backend-store-uri sqlite:///mlflow_data/mlflow.db \
    --default-artifact-root ./mlflow_data/mlartifacts
```

L'interface est ensuite consultable dans un navigateur à l'adresse
http://127.0.0.1:5001. Voir `demos/README.md` pour le détail, et
`speech/ressources-techniques.md` pour son rôle dans la stack technique.

Si aucun serveur n'est joignable (par exemple sur Kaggle Notebooks ou Colab, qui ne
voient pas votre machine locale), remplacez la ligne `mlflow.set_tracking_uri(...)`
ci-dessous par `mlflow.set_tracking_uri("file:./mlruns")` : MLflow écrit alors ses runs
dans un simple dossier local, consultable plus tard avec `mlflow ui`.


In [3]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5001")
experience = mlflow.set_experiment("OFDS - Jour 2 - Titanic")
print("Expérience MLflow active :", experience.name)


2026/09/16 17:40:55 INFO mlflow.tracking.fluent: Experiment with name 'OFDS - Jour 2 - Titanic' does not exist. Creating a new experiment.


Expérience MLflow active : OFDS - Jour 2 - Titanic


In [4]:
with mlflow.start_run(run_name="logreg_features_enrichies"):
    model = LogisticRegression(max_iter=5000)
    model.fit(X_train, y_train)

    scores = cross_val_score(model, X_train, y_train, cv=5)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("features", list(X.columns))
    mlflow.log_metric("cv_accuracy_mean", scores.mean())
    mlflow.log_metric("cv_accuracy_std", scores.std())

# Premier modele du notebook 01 : memes passagers, memes plis, 5 colonnes sans valeurs manquantes
features_premier_modele = ["Pclass", "Sex_male", "SibSp", "Parch", "Fare"]
scores_premier_modele = cross_val_score(
    LogisticRegression(max_iter=5000),
    titanic.loc[X_train.index, features_premier_modele], y_train, cv=5,
)

print("Premier modèle (notebook 01) — score moyen CV :", round(scores_premier_modele.mean(), 3))
print("Modèle enrichi — scores par pli :", scores.round(3))
print("Modèle enrichi — score moyen CV :", round(scores.mean(), 3))

View run logreg_features_enrichies at: http://127.0.0.1:5001/#/experiments/1/runs/902c09a4cdfc4681b4c9cdc04ca22dde
View experiment at: http://127.0.0.1:5001/#/experiments/1


Premier modèle (notebook 01) — score moyen CV : 0.798
Modèle enrichi — scores par pli : [0.811 0.818 0.81  0.739 0.838]
Modèle enrichi — score moyen CV : 0.803


**Ce qu'on observe** : les deux modèles sont évalués sur les mêmes passagers et les mêmes
plis de cross-validation, la comparaison est donc équitable. Le modèle enrichi fait à peine
mieux que le premier modèle (__CV_ENRICHI__ contre __CV_PREMIER__), un écart bien plus petit que la
variation d'un pli à l'autre. Pour une régression logistique, ces nouvelles variables
apportent peu : `FamilySize` résume `SibSp` et `Parch`, que le premier modèle utilisait déjà
séparément. Les méthodes ensemblistes (notebooks 06 et 07), qui captent les interactions
entre variables, en tirent davantage parti.

Ce score de cross-validation est enregistré dans le run MLflow `logreg_features_enrichies`,
consultable et comparable à tout moment.

## Étape 2 — Recherche d'hyperparamètres avec GridSearchCV

**Question de réflexion :** quels hyperparamètres de la régression logistique pourrait-on
faire varier, et comment être sûr de ne pas choisir la meilleure combinaison « par
chance » sur un seul découpage ?

On fait varier `C` (l'inverse de la force de régularisation) et le type de pénalité, Ridge
(`l1_ratio=0`) ou Lasso (`l1_ratio=1`), explorés à la main au notebook 02. `l1_ratio`
remplace l'ancien paramètre `penalty="l2"` / `"l1"`, déprécié depuis scikit-learn 1.8.


In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "l1_ratio": [0, 1],
}

with mlflow.start_run(run_name="gridsearch_logreg"):
    grid = GridSearchCV(
        LogisticRegression(solver="liblinear", max_iter=5000),
        param_grid,
        cv=5,
        scoring="accuracy",
    )
    grid.fit(X_train, y_train)

    mlflow.log_param("model", "LogisticRegression (GridSearchCV)")
    mlflow.log_params(grid.best_params_)
    mlflow.log_metric("cv_accuracy_mean", grid.best_score_)

print("Meilleurs paramètres :", grid.best_params_)
print("Meilleur score en cross-validation :", round(grid.best_score_, 3))


View run gridsearch_logreg at: http://127.0.0.1:5001/#/experiments/1/runs/e3c657ce55ef4f18a255aff16dc0d4b2
View experiment at: http://127.0.0.1:5001/#/experiments/1


Meilleurs paramètres : {'C': 10, 'l1_ratio': 0}
Meilleur score en cross-validation : 0.805


**Ce qu'on observe** : `GridSearchCV` évalue chaque combinaison de la grille sur les 5
plis de cross-validation, et retient celle qui donne la meilleure performance moyenne.
Pour une grille plus large, `RandomizedSearchCV` échantillonnerait aléatoirement les
combinaisons plutôt que de toutes les tester. Le run MLflow `gridsearch_logreg` enregistre
directement `grid.best_params_` comme paramètres : plus besoin de les recopier à la main
pour retrouver la configuration gagnante.


## Étape 3 — Soumission sur Kaggle

**Question de réflexion :** quelles sont les colonnes attendues dans un fichier de
soumission Kaggle pour la compétition Titanic ?


In [6]:
best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": titanic.loc[X_test.index, "PassengerId"],
    "Survived": predictions,
})
submission.to_csv("submission.csv", index=False)

submission.head()


,PassengerId,Survived
709,710,0
439,440,0
840,841,0
720,721,1
39,40,1


**Ce qu'on observe** : le fichier `submission.csv` contient exactement les deux colonnes
attendues par Kaggle, `PassengerId` et `Survived`. Ici, `X_test` provient de notre propre
split local (nous connaissons donc déjà `y_test`, ce qui nous sert de vérification) ;
pour une vraie soumission Kaggle, on applique `best_model.predict()` sur le `test.csv`
officiel, qui ne contient pas la colonne `Survived`.


In [7]:
from sklearn.metrics import accuracy_score

# Verification locale, possible ici uniquement parce que nous connaissons y_test
test_accuracy = accuracy_score(y_test, predictions)

with mlflow.start_run(run_name="final_model_evaluation"):
    mlflow.log_params(grid.best_params_)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.sklearn.log_model(best_model, name="model")
    mlflow.log_artifact("submission.csv")

print("Accuracy sur notre jeu de test local :", round(test_accuracy, 3))


View run final_model_evaluation at: http://127.0.0.1:5001/#/experiments/1/runs/e6a8ba5590974b358abfe50b279b1639
View experiment at: http://127.0.0.1:5001/#/experiments/1
Accuracy sur notre jeu de test local : 0.81


**À retenir pour le débriefing du TP** : trois scores de cross-validation se suivent
maintenant, celui du premier modèle (notebook 01), celui du modèle enrichi, puis celui du
modèle réglé par `GridSearchCV`. Avec une régression logistique, les gains restent modestes :
le réglage des hyperparamètres pèse peu, et les nouvelles variables ne donnent leur pleine
mesure qu'avec des modèles capables d'exploiter les interactions entre variables (notebooks
06 et 07).

Ouvrez http://127.0.0.1:5001 : l'expérience « OFDS - Jour 2 - Titanic » contient
désormais trois runs (`logreg_features_enrichies`, `gridsearch_logreg`,
`final_model_evaluation`) comparables côte à côte, avec le modèle et le fichier de
soumission attachés en artefacts au dernier run.